Functions we need in our method

In [ ]:
#pip install mealpy which includes JADE
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from mealpy import FloatVar,DE
# Sigmoid function
def S(v,e0,r,v0):
    '''Sigmoid function for the Jansen-Rit model'''
    v0 = v0
    return 2 * e0 / (1 + np.exp(r * (v0 - v)))

# ODE subsystem
def jansen_rit_ode_part(y,paramers):
    '''Deterministic part for the Jansen-Rit model'''
    a, b= paramers
    y0, y1, y2, y3, y4, y5 = y
    dy0 = y3
    dy1 = y4
    dy2 = y5
    dy3 = - 2 * a * y3 - a**2 * y0
    dy4 = - 2 * a * y4 - a**2 * y1 
    dy5 = - 2 * b * y5 - b**2 * y2
    return np.array([dy0, dy1, dy2, dy3, dy4, dy5])

# SDE subsystem drift term
def jansen_rit_sde_part_drift(y,p,paramers):
    '''Stochastic part for the Jansen-Rit model'''
    A,a,B,b,C,K,e0,r,v0= paramers
    C1 = C          
    C2 = 0.8 * C  
    C3 = 0.25 * C
    C4 = 0.25 * C

    y0, y1, y2, y3, y4, y5 = y
    dy0 = 0
    dy1 = 0
    dy2 = 0
    dy3 = A * a * S(y1 - y2,e0,r,v0)
    dy4 = A * a *  C2 * S(C1 * y0,e0,r,v0) + A * a * K * S(p,e0,r,v0)
    dy5 = B * b * C4 * S(C3 * y0,e0,r,v0)
    return np.array([dy0, dy1, dy2, dy3, dy4, dy5])

# SDE subsystem diffusion term
def jansen_rit_diffusion(diff):
    '''Diffusion term for the Jansen-Rit model SDEs'''
    return np.array([0,0,0,0, diff, 0])  # 简单起见，我们假设扩散系数是常数


def strang_step(y, h, p, params,diff):
    '''Strang splitting step for the Jansen-Rit model with SDEs and ODEs'''
    diff = diff

    A,a,B,b,C,K,e0,r,v0 = params
    PARAMER = np.array([a, b])
    parameter = np.array([A,a,B,b,C,K,e0,r,v0])
    # Half step for A
    y_half_A = y + (h/2) * jansen_rit_ode_part(y,PARAMER)
    # Full step for B
    g = jansen_rit_diffusion(diff)
    dW_node1 = np.sqrt(h) * np.random.randn(6)
    y_full_B = y_half_A + h * jansen_rit_sde_part_drift(y_half_A,p,parameter)  + g * dW_node1 
    # Half step for A again
    y_full_A = y_full_B + (h/2) * jansen_rit_ode_part(y_full_B,PARAMER)
    return y_full_A


###Chen-Fliess series expansion of x0,x2,x3,x5
def fun_model_x2_generate(C,B,v0,q0,q2,q3,q5,t0,t):
    '''Generate x2 using Chen-Fliess series expansion'''
    e0 = 5
    r = 0.56
    b = 50
    Sig = np.exp(r*(-0.25*C*q0+v0))
    L = q2 + q5*(t-t0) + (0.25*B*e0*b*C/(1+Sig)-b**2*q2-2*b*q5)*((t-t0)**2/2)+(0.0625*B*e0*b*C**2*r*q3*Sig/(1+Sig)**2-b**2*q5-2*b*(0.25*B*e0*b*C/(1+Sig)-b**2*q2-2*b*q5))*((t-t0)**3/6)
    return L

def fun_model_x5_generate(C,B,v0,q0,q2,q3,q5,t0,t):
    '''Generate x5 using Chen-Fliess series expansion'''
    e0 = 5
    r = 0.56
    b = 50
    Sig = np.exp(r*(-0.25*C*q0+v0))
    L = q5 + (0.25*B*e0*b*C/(1+Sig)-b**2*q2-2*b*q5)*(t-t0)+(0.0625*B*e0*b*C**2*r*q3*Sig/(1+Sig)**2-b**2*q5-2*b*(0.25*B*e0*b*C/(1+Sig)-b**2*q2-2*b*q5))*((t-t0)**2/2)
    return L


def fun_model_x0_generate(A,v0,q0,q3,y1,y2,t0,t):
    '''Generate x0 using Chen-Fliess series expansion'''
    r = 0.56
    e0 = 5
    a = 100
    Sig = np.exp(r*(v0-y1))
    L = q0 + q3*(t-t0) + ((A*e0*a)/(1+Sig)-a**2*q0-2*a*q3)*((t-t0)**2/2)+(A*e0*a*r*Sig*(y2)/(Sig+1)**2-a**2*q3-2*a*((A*e0*a)/(1+Sig)-a**2*q0-2*a*q3))*((t-t0)**3/6)
    return L


def fun_model_x3_generate(A,v0,q0,q3,y1,y2,t0,t):
    '''Generate x3 using Chen-Fliess series expansion'''
    r = 0.56
    e0 = 5
    a = 100
    Sig = np.exp(r*(v0-y1))
    L = q3 + ((A*e0*a)/(1+Sig)-a**2*q0-2*a*q3)*(t-t0)+(A*e0*a*r*Sig*(y2)/(Sig+1)**2-a**2*q3-2*a*((A*e0*a)/(1+Sig)-a**2*q0-2*a*q3))*((t-t0)**2/2)
    return L


#############################################################################
#####Loss function

def fun_model_output(A,B,C,v0, K,q,y_1,y_2,t0,t):
    '''Model output using Chen-Fliess series expansion--second order'''
    a = 100
    b = 50
    e0 = 5
    r = 0.56
    y1,y2 = y_1; y3,y4 = y_2; q0, q1, q2, q3, q4, q5= q
    Sig1 = np.exp(r*(v0-y3));Sig_x2 = np.exp(r*(-C*q0+v0));  Sig_x3 = np.exp(r*(-0.25*C*q0+v0))
    L = y1 + y2*(t-t0)+(0.8*A*e0*a*C/(1+Sig_x2)+A*e0*a*K/(1+Sig1)-a**2*q1-2*a*q4-(0.25*B*e0*b*C/(1+Sig_x3)-b**2*q2-2*b*q5))*((t-t0)**2/2)
    #+(0.8*A*e0*a*C**2*r*q3*Sig_x2/(1+Sig_x2)**2+r*A*e0*a*K*y4*Sig1/(1+Sig1)**2-a**2*q4-2*a*(0.8*A*e0*a*C/(1+Sig_x2)-a**2*q1-2*a*q4)
    #  -(0.0625*B*e0*b*C**2*r*q3*Sig_x3/(1+Sig_x3)**2-b**2*q5-2*b*(0.25*B*e0*b*C/(1+Sig_x3)-b**2*q2-2*b*q5)))*((t-t0)**3/6))
    return L


def fun_model_output_third(A,B,C,v0, K,q,y_1,y_2,t0,t):
    '''Model output using Chen-Fliess series expansion--third order'''
    a = 100
    b = 50
    e0 = 5
    r = 0.56
    y1,y2 = y_1; y3,y4 = y_2; q0, q1, q2, q3, q4, q5= q
    Sig1 = np.exp(r*(v0-y3));Sig_x2 = np.exp(r*(-C*q0+v0));  Sig_x3 = np.exp(r*(-0.25*C*q0+v0))
    L = (y1 + y2*(t-t0)+(0.8*A*e0*a*C/(1+Sig_x2)+A*e0*a*K/(1+Sig1)-a**2*q1-2*a*q4-(0.25*B*e0*b*C/(1+Sig_x3)-b**2*q2-2*b*q5))*((t-t0)**2/2)
    +(0.8*A*e0*a*C**2*r*q3*Sig_x2/(1+Sig_x2)**2+r*A*e0*a*K*y4*Sig1/(1+Sig1)**2-a**2*q4-2*a*(0.8*A*e0*a*C/(1+Sig_x2)-a**2*q1-2*a*q4)
      -(0.0625*B*e0*b*C**2*r*q3*Sig_x3/(1+Sig_x3)**2-b**2*q5-2*b*(0.25*B*e0*b*C/(1+Sig_x3)-b**2*q2-2*b*q5)))*((t-t0)**3/6))
    return L


def objective_function_re(params,N_interval,y_iniii_node1,y_iniii_node2,t_spans, mean_sol_data_true):
    A,t1,t2,v0,t3 = params
    B = A/t1; C = t2/A; K = t3/A
    sol_fitted = []
    qq = []
    q_ini =  np.random.uniform(0,0,6)
    for i in range(N_interval):
        y_135 = mean_sol_data_true[:,i]
        q = q_ini
        q[1] = y_iniii_node1[i,0]+q[2];q[4] = y_iniii_node1[i,1]+q[5] 
        qq.append(q)
        y12 = y_iniii_node1[i]
        y_12 = y_iniii_node2[i]
        t_span31 = t_spans[i,]
        
        fun =  (fun_model_output(A,B,C,v0, K,q,y12, y_12,t_span31[0], t_span31[-1])-y_135[-1])**2
        sol_fitted.append(fun)   
        q0 = fun_model_x0_generate(A,v0,q[0],q[3],y12[0],y12[1],t_span31[0], t_span31[-1])
        q2 = fun_model_x2_generate(C,B,v0,q[0],q[2],q[3],q[5],t_span31[0], t_span31[-1])
        q3 = fun_model_x3_generate(A,v0,q[0],q[3],y12[0],y12[1],t_span31[0], t_span31[-1])
        q5 = fun_model_x5_generate(C,B,v0,q[0],q[2],q[3],q[5],t_span31[0], t_span31[-1])
        q_ini = [q0,0,q2,q3,0,q5]
    fun_epoch = np.array(sol_fitted)
    return np.sqrt(sum(fun_epoch[255:])) # only consider the loss after 255 intervals; sum over all intervals; then take the square root
    

Upload data

In [ ]:
data_E = np.zeros((18,2560*2,6)) #18 channels, each with 2x10s--> 20s, repeated 6 times
data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_9_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_9_2.txt', delimiter=',')
data_E[:,:,0] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_13_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_13_2.txt', delimiter=',')
data_E[:,:,1] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_21_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_21_2.txt', delimiter=',')
data_E[:,:,2] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_36_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_36_2.txt', delimiter=',')
data_E[:,:,3] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_38_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_38_2.txt', delimiter=',')
data_E[:,:,4] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data1 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_40_1.txt', delimiter=',')
data2 = np.loadtxt('Z:/eeg_jansen/EEG/segment data/Epileptic/Epileptic/P3/P3_40_2.txt', delimiter=',')
data_E[:,:,5] = np.concatenate((data1[:,:2560],data2[:,:2560]),axis=1)

data_all = data_E*10 #data scaling
data = data_all
dt = 1/256  #sampling rate
################
##channel pairs for input-output mapping, which can be recoded in an array.  Note that cahnnel paris are specified for different datasets
channel_pairs = [
    (0, 3), (3, 0),
    (1, 5), (5, 1),
    (2, 6), (6, 2),
    (4, 7), (7, 4),
    (1, 7), (7, 1),
    (3, 5), (5, 3),
    (8, 12), (12, 8),
    (13, 9), (9, 13),
    (14, 10), (10, 14),
    (15, 11), (11, 15),
    (9, 14), (14, 9),
    (16, 9), (9, 16),
    (17, 10), (10, 17),
    (16, 13), (13, 16),
    (17, 14), (14, 17),
    (4, 16), (16, 4),
    (0, 16), (16, 0),
    (6, 17), (17, 6),
    (2, 17), (17, 2),
    (7, 17), (17, 7),
    (3, 17), (17, 3),
    (7, 16), (16, 7),
    (3, 16), (16, 3),
    (0, 17), (17, 0),
    (17, 4), (4, 17),
    (4, 12), (12, 4),
    (0, 8), (8, 0),
    (5, 13), (13, 5),
    (6, 14), (14, 6),
    (7, 15), (15, 7),
    (1, 9), (9, 1),
    (2, 10), (10, 2),
    (3, 11), (11, 3),
    (4, 14), (14, 4),
    (0, 10), (10, 0),
    (4, 8), (8, 4),
    (0, 12), (12, 0),
    (5, 9), (9, 5),
    (1, 13), (13, 1),
    (6, 10), (10, 6),
    (2, 14), (14, 2),
    (7, 11), (11, 7),
    (3, 15), (15, 3),
    (5, 10), (10, 5),
    (2, 13), (13, 2),
    (12, 11), (11, 12),
    (8, 15), (15, 8),
    (12, 15), (15, 12),
    (8, 11), (11, 8),
    (4, 15), (15, 4),
    (0, 11), (11, 0),
    (16, 15), (15, 16),
    (12, 10), (10, 12),
    (8, 14), (14, 8),
    (0, 1), (1, 0),
    (0, 2), (2, 0),
    (0, 4), (4, 0),
    (0, 5), (5, 0),
    (0, 6), (6, 0),
    (0, 7), (7, 0),
    (0, 9), (9, 0),
    (0, 13), (13, 0),
    (0, 14), (14, 0),
    (0, 15), (15, 0),
    (1, 2), (2, 1),
    (1, 3), (3, 1),
    (1, 4), (4, 1),
    (1, 6), (6, 1),
    (1, 8), (8, 1),
    (1, 10), (10, 1),
    (1, 11), (11, 1),
    (1, 12), (12, 1),
    (1, 14), (14, 1),
    (1, 15), (15, 1),
    (1, 16), (16, 1),
    (1, 17), (17, 1),
 (2, 3), (3, 2),
 (2, 4), (4, 2),
 (2, 5), (5, 2),
 (2, 7), (7, 2),
 (2, 8), (8, 2),
 (2, 9), (9, 2),
 (2, 11), (11, 2),
 (2, 12), (12, 2),
 (2, 15), (15, 2),
 (2, 16), (16, 2),
 (3, 4), (4, 3),
 (3, 6), (6, 3),
 (3, 7), (7, 3),
 (3, 8), (8, 3),
 (3, 9), (9, 3),
 (3, 10), (10, 3),
 (3, 12), (12, 3),
 (3, 13), (13, 3),
 (3, 14), (14, 3),
 (4, 5), (5, 4),
 (4, 6), (6, 4),
 (4, 9), (9, 4),
 (4, 10), (10, 4),
 (4, 11), (11, 4),
 (4, 13), (13, 4),
 (5, 6), (6, 5),
 (5, 7), (7, 5),
 (5, 8), (8, 5),
 (5, 11), (11, 5),
 (5, 12), (12, 5),
 (5, 14), (14, 5),
 (5, 15), (15, 5),
 (5, 16), (16, 5),
 (5, 17), (17, 5),
 (6, 7), (7, 6),
 (6, 8), (8, 6),
 (6, 9), (9, 6),
 (6, 11), (11, 6),
 (6, 12), (12, 6),
 (6, 13), (13, 6),
 (6, 15), (15, 6),
 (6, 16), (16, 6),
 (7, 8), (8, 7), 
 (7, 9), (9, 7), 
 (7, 10), (10, 7), 
 (7, 12), (12, 7), 
 (7, 13), (13, 7), 
 (7, 14), (14, 7),
 (8, 9), (9, 8),
 (8, 10), (10, 8),
 (8, 13), (13, 8),
 (8, 16), (16, 8),
 (8, 17), (17, 8),
 (9, 10), (10, 9),
 (9, 11), (11, 9),
 (9, 12), (12, 9),
 (9, 15), (15, 9),
 (9, 17), (17, 9),
 (10, 11), (11, 10),
 (10, 13), (13, 10),
 (10, 15), (15, 10),
 (10, 16), (16, 10),
 (11, 13), (13, 11),
 (11, 14), (14, 11),
 (11, 16), (16, 11),
 (11, 17), (17, 11),
 (12, 13), (13, 12),
 (12, 14), (14, 12),
 (12, 16), (16, 12),
 (12, 17), (17, 12),
 (13, 14), (14, 13),
 (13, 15), (15, 13),
 (13, 17), (17, 13),
 (14, 15), (15, 14),
 (14, 16), (16, 14),
 (15, 17), (17, 15),
 (16, 17), (17, 16),

]
###prepare input-output data based on channel pairs
data_input = []
data_output = []

for ch_in, ch_out in channel_pairs:
    data_input.append(data[ch_in])
    data_output.append(data[ch_out])
data_input = np.concatenate(data_input,axis=1)
data_output = np.concatenate(data_output,axis=1)
input_data_ill = data_output
T = len(input_data_ill)

split_size = 2 ### the length of each epoch
n_interval = split_size
n_reptation = input_data_ill.shape[1]

estimate_value = np.zeros((n_reptation,5))
print(T,n_reptation)

5120
1836


Estimation procedure

In [ ]:
for mm in range(n_reptation):
    observation_output = data_output[:,mm]
    observation_input = data_input[:,mm]
    step = split_size - 1 ### overlapping subarrays, the end of one subarray is the start of the next subarray
    def create_subarrays(data):
        '''segment data into subarrays'''
        return [data[i:i + split_size] for i in range(0, len(data) - split_size + 1, step)]
    subarrays_output = create_subarrays(observation_output)
    subarrays_input = create_subarrays(observation_input)
    sol_output = np.stack(subarrays_output, axis=1)        # shape: (split_size, n_parts)

    sol_input = np.stack(subarrays_input, axis=1)    # shape: (split_size, n_parts)
    h = dt #discretization step
    y_iniii_input = np.zeros((len(subarrays_output),2))
    y_iniii_input[:,0] = sol_input[0, :]
    y_iniii_input[:,1] = (sol_input[1,:]-sol_input[0,:])/h

    y_iniii_output = np.zeros((len(subarrays_output),2))
    y_iniii_output[:,0] = sol_output[0, :]
    y_iniii_output[:,1] = (sol_output[1,:]-sol_output[0,:])/h
    t_last = len(observation_output)*h
    t_value = np.arange(0, t_last, h)
    aa = sol_input.shape
    N_interval = aa[1]
    t_spans = np.zeros((N_interval, n_interval))
    t_spans[0] = t_value[:n_interval]
    for i in range(1, N_interval):
        t_spans[i, 0] = t_spans[i-1, -1]
        t_spans[i, 1:] = t_value[n_interval + (i-1)*(n_interval-1) : n_interval + i*(n_interval-1)]
    
    
    def wrapped_function(params):
        return objective_function_re(params,N_interval,y_iniii_output,y_iniii_input, t_spans, sol_output)
    problem_dict = {
    "obj_func": wrapped_function,
    "bounds": FloatVar(lb=(2.6,0.05,0,0,0), ub=(6,1,1000,10,700),name='delta'),
    "minmax": "min",
    "verbose": False,
}

    model = DE.JADE(epoch=250, pop_size = 15,  verbose = False)
    g_best = model.solve(problem_dict,n_workers=-1)
    estimate_value[mm] = g_best.solution



    
    

    

        
   

2026/01/20 01:04:06 PM, INFO, mealpy.evolutionary_based.DE.JADE: JADE(epoch=250, pop_size=15, miu_f=0.5, miu_cr=0.5, pt=0.1, ap=0.1)
2026/01/20 01:04:09 PM, INFO, mealpy.evolutionary_based.DE.JADE: >>>Problem: P, Epoch: 1, Current best: 195.44656465460153, Global best: 195.44656465460153, Runtime: 1.17172 seconds
2026/01/20 01:04:10 PM, INFO, mealpy.evolutionary_based.DE.JADE: >>>Problem: P, Epoch: 2, Current best: 190.50128315194297, Global best: 190.50128315194297, Runtime: 1.17238 seconds
2026/01/20 01:04:11 PM, INFO, mealpy.evolutionary_based.DE.JADE: >>>Problem: P, Epoch: 3, Current best: 190.50128315194297, Global best: 190.50128315194297, Runtime: 1.19062 seconds
2026/01/20 01:04:12 PM, INFO, mealpy.evolutionary_based.DE.JADE: >>>Problem: P, Epoch: 4, Current best: 190.30559334693848, Global best: 190.30559334693848, Runtime: 1.11775 seconds
2026/01/20 01:04:13 PM, INFO, mealpy.evolutionary_based.DE.JADE: >>>Problem: P, Epoch: 5, Current best: 190.24868426157414, Global best: 19

Save estimation results

In [ ]:
estimate_median =  np.round(estimate_value,2)

column_names = ['A','A/B','A*C','v0','A*K']
estimate_q = pd.DataFrame(estimate_median,columns=column_names)

print(estimate_q)

folder_path = 'Z:/Jansen-Rit-Coupled/real_median'

file_path = os.path.join(folder_path,'Dif_ESTIMATE_REAL_P3.csv')

estimate_q.to_csv(file_path,index=False)


